# **TikTok Scraper Notebook for HealthPH+**


# **Dependencies**

In [1]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [2]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc

In [3]:
print("âœ… Libraries loaded successfully")

âœ… Libraries loaded successfully


## Tiktok

In [4]:
# TikTok scraper config (Playwright, no Apify)
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "docs" / "keywords").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "docs" / "keywords").exists():
    raise FileNotFoundError(f"Could not locate project root from: {Path.cwd()}")

TT_KEYWORDS_FILE = PROJECT_ROOT / "docs/keywords/ri_keywords.csv"
TT_KEYWORD_COLUMN = None  # None means first CSV column

TT_MAX_VIDEOS_PER_KEYWORD = 30
TT_SCROLL_ROUNDS = 20
TT_SCROLL_WAIT_MS = 1500
TT_HEADLESS = True
TT_NAV_TIMEOUT_MS = 60000

TT_START_DATE = pd.Timestamp("2025-01-01", tz="UTC")
TT_END_DATE = pd.Timestamp.now(tz="UTC")

TT_OUTPUT_DIR = PROJECT_ROOT / "data/raw/tiktok"
TT_OUTPUT_FILE = TT_OUTPUT_DIR / f"tiktok_notebook_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.csv"

print("Project root:", PROJECT_ROOT)
print("Keywords file:", TT_KEYWORDS_FILE)
print("Date range:", TT_START_DATE, "to", TT_END_DATE)
print("Output:", TT_OUTPUT_FILE)


Keywords file: ..\docs\keywords\ri_keywords.csv
Date range: 2025-01-01 00:00:00+00:00 to 2026-03-03 08:10:35.124582+00:00
Output: ..\data\raw\tiktok\tiktok_notebook_2026-03-03_16-10-35.csv


In [5]:
import asyncio


def _safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _to_utc_datetime(value: Any) -> pd.Timestamp | None:
    if value is None or value == "":
        return None
    ts = pd.to_datetime(value, errors="coerce", utc=True)
    if pd.isna(ts):
        return None
    return ts


def _contains_keyword(text: str, keywords: list[str]) -> bool:
    hay = _safe_text(text).lower()
    return any(_safe_text(keyword).lower() in hay for keyword in keywords if _safe_text(keyword))


def _load_keywords_from_csv(filepath: Path, column: str | None = None) -> list[str]:
    if not filepath.exists():
        raise FileNotFoundError(f"Keyword file not found: {filepath}")

    df = pd.read_csv(filepath)
    if df.empty:
        return []

    col = column if column in df.columns else df.columns[0]
    values = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.strip("\"'")
    )
    keywords = [value for value in values.tolist() if value]
    # Keep order while removing duplicates
    deduped = list(dict.fromkeys(keywords))
    return deduped


def _extract_video_id(url: str) -> str | None:
    if not isinstance(url, str):
        return None
    match = re.search(r"/video/(\d+)", url)
    if match:
        return match.group(1)
    return None


def _stable_tiktok_id(video_url: str, created_at: str, caption: str) -> str:
    direct_id = _extract_video_id(video_url)
    if direct_id:
        return direct_id
    base = f"{video_url}|{created_at}|{caption[:120]}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()


def _extract_json_blobs_from_html(html: str) -> list[dict[str, Any]]:
    blobs: list[dict[str, Any]] = []
    if not html:
        return blobs

    # 1) Application/json script tags commonly used by TikTok.
    script_patterns = [
        r'<script[^>]*id="__UNIVERSAL_DATA_FOR_REHYDRATION__"[^>]*>(.*?)</script>',
        r'<script[^>]*id="SIGI_STATE"[^>]*>(.*?)</script>',
    ]
    for pattern in script_patterns:
        for match in re.finditer(pattern, html, flags=re.DOTALL):
            raw = _safe_text(match.group(1))
            if not raw:
                continue
            try:
                parsed = json.loads(raw)
                if isinstance(parsed, dict):
                    blobs.append(parsed)
            except json.JSONDecodeError:
                continue

    # 2) Inline JS assignments such as window["SIGI_STATE"] = {...};
    assign_patterns = [
        r'window\["SIGI_STATE"\]\s*=\s*(\{.*?\})\s*;',
        r'window\.__UNIVERSAL_DATA_FOR_REHYDRATION__\s*=\s*(\{.*?\})\s*;',
    ]
    for pattern in assign_patterns:
        for match in re.finditer(pattern, html, flags=re.DOTALL):
            raw = _safe_text(match.group(1))
            if not raw:
                continue
            try:
                parsed = json.loads(raw)
                if isinstance(parsed, dict):
                    blobs.append(parsed)
            except json.JSONDecodeError:
                continue

    return blobs


def _find_first_matching_payload(node: Any) -> dict[str, Any] | None:
    if isinstance(node, dict):
        has_caption = "desc" in node and _safe_text(node.get("desc")) != ""
        has_time = "createTime" in node
        stats = node.get("stats")
        has_stats = isinstance(stats, dict) and ("diggCount" in stats or "shareCount" in stats)
        if has_caption or (has_time and has_stats):
            return node
        for value in node.values():
            result = _find_first_matching_payload(value)
            if result is not None:
                return result

    if isinstance(node, list):
        for value in node:
            result = _find_first_matching_payload(value)
            if result is not None:
                return result

    return None


def _parse_tiktok_video_payload(blobs: list[dict[str, Any]]) -> dict[str, Any]:
    for blob in blobs:
        payload = _find_first_matching_payload(blob)
        if payload is not None:
            return payload
    return {}


def _extract_transcript_native(payload: dict[str, Any]) -> str:
    chunks: list[str] = []

    subtitle_candidates: list[Any] = []
    if isinstance(payload, dict):
        subtitle_candidates.append(payload.get("subtitleInfos"))
        subtitle_candidates.append(payload.get("subtitles"))
        video_obj = payload.get("video") if isinstance(payload.get("video"), dict) else {}
        subtitle_candidates.append(video_obj.get("subtitleInfos"))
        subtitle_candidates.append(video_obj.get("subtitles"))

    def _walk(node: Any) -> None:
        if isinstance(node, dict):
            for key in ("text", "content", "value"):
                candidate = _safe_text(node.get(key))
                if candidate:
                    chunks.append(candidate)
            for value in node.values():
                _walk(value)
        elif isinstance(node, list):
            for value in node:
                _walk(value)

    for candidate in subtitle_candidates:
        _walk(candidate)

    if not chunks:
        return ""

    transcript = " ".join(chunks)
    transcript = re.sub(r"\s+", " ", transcript).strip()
    return transcript


def _normalize_tiktok_row(video_url: str, payload: dict[str, Any], matched_keyword: str) -> dict[str, Any]:
    caption = _safe_text(payload.get("desc"))
    created_raw = payload.get("createTime")
    created_at = ""
    if created_raw is not None and _safe_text(created_raw).isdigit():
        ts = pd.to_datetime(int(created_raw), unit="s", utc=True, errors="coerce")
        if not pd.isna(ts):
            created_at = ts.strftime("%Y-%m-%d %H:%M:%S UTC")
    else:
        parsed = _to_utc_datetime(created_raw)
        if parsed is not None:
            created_at = parsed.strftime("%Y-%m-%d %H:%M:%S UTC")

    stats = payload.get("stats") if isinstance(payload.get("stats"), dict) else {}
    like_count = pd.to_numeric(stats.get("diggCount"), errors="coerce")
    share_count = pd.to_numeric(stats.get("shareCount"), errors="coerce")
    like_value = None if pd.isna(like_count) else int(like_count)
    share_value = None if pd.isna(share_count) else int(share_count)

    transcript = _extract_transcript_native(payload)
    text = re.sub(r"\s+", " ", f"{caption} {transcript}").strip()

    author = payload.get("author") if isinstance(payload.get("author"), dict) else {}
    author_username = _safe_text(author.get("uniqueId") or payload.get("authorUniqueId"))

    row_id = _stable_tiktok_id(video_url=video_url, created_at=created_at, caption=caption)
    return {
        "created_at": created_at,
        "id": row_id,
        "text": text,
        "source": "tiktok",
        "url": video_url,
        "caption": caption,
        "transcript": transcript,
        "like_count": like_value,
        "share_count": share_value,
        "matched_keyword": matched_keyword,
        "author_username": author_username,
    }


async def _collect_video_links_for_keyword(page, keyword: str, scroll_rounds: int, scroll_wait_ms: int) -> list[str]:
    search_url = f"https://www.tiktok.com/search/video?q={quote_plus(keyword)}"
    await page.goto(search_url, wait_until="domcontentloaded")

    links: set[str] = set()
    for _ in range(scroll_rounds):
        anchors = await page.eval_on_selector_all(
            "a[href*='/video/']",
            "els => els.map(el => el.href).filter(Boolean)",
        )
        for href in anchors:
            if isinstance(href, str) and "/video/" in href:
                links.add(href.split("?")[0])

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(scroll_wait_ms)

    return sorted(links)


async def _scrape_tiktok_keywords_async(
    keywords: list[str],
    max_videos_per_keyword: int = 30,
    scroll_rounds: int = 20,
    scroll_wait_ms: int = 1500,
    headless: bool = True,
    nav_timeout_ms: int = 60000,
    start_date: pd.Timestamp | None = None,
    end_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen_ids: set[str] = set()

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=headless)
        context = await browser.new_context()
        page = await context.new_page()
        page.set_default_timeout(nav_timeout_ms)

        for keyword in keywords:
            keyword = _safe_text(keyword)
            if not keyword:
                continue

            print(f"\nScraping keyword: {keyword}")
            try:
                video_links = await _collect_video_links_for_keyword(
                    page=page,
                    keyword=keyword,
                    scroll_rounds=scroll_rounds,
                    scroll_wait_ms=scroll_wait_ms,
                )
            except PlaywrightTimeoutError:
                print(f"  ! Timeout during search for keyword: {keyword}")
                continue
            except Exception as exc:
                print(f"  ! Failed search for keyword {keyword}: {exc}")
                continue

            kept = 0
            for video_url in video_links:
                if kept >= max_videos_per_keyword:
                    break

                try:
                    await page.goto(video_url, wait_until="domcontentloaded")
                    html = await page.content()
                except PlaywrightTimeoutError:
                    print(f"  ! Timeout opening video: {video_url}")
                    continue
                except Exception:
                    continue

                blobs = _extract_json_blobs_from_html(html)
                payload = _parse_tiktok_video_payload(blobs)
                if not payload:
                    continue

                row = _normalize_tiktok_row(video_url=video_url, payload=payload, matched_keyword=keyword)
                if not row["id"]:
                    continue

                if row["id"] in seen_ids:
                    continue

                if not _contains_keyword(f"{row['caption']} {row['transcript']}", [keyword]):
                    continue

                post_time = _to_utc_datetime(row["created_at"])
                if start_date is not None and post_time is not None and post_time < start_date:
                    continue
                if end_date is not None and post_time is not None and post_time > end_date:
                    continue

                rows.append(row)
                seen_ids.add(row["id"])
                kept += 1

            print(f"  + Kept {kept} video(s)")

        await context.close()
        await browser.close()

    columns = [
        "created_at",
        "id",
        "text",
        "source",
        "url",
        "caption",
        "transcript",
        "like_count",
        "share_count",
        "matched_keyword",
        "author_username",
    ]

    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows, columns=columns)


def _scrape_tiktok_keywords_sync_runner(
    keywords: list[str],
    max_videos_per_keyword: int = 30,
    scroll_rounds: int = 20,
    scroll_wait_ms: int = 1500,
    headless: bool = True,
    nav_timeout_ms: int = 60000,
    start_date: pd.Timestamp | None = None,
    end_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    # Run Playwright on a dedicated event loop in a worker thread.
    # This avoids Windows/Jupyter loop limitations with subprocess transport.
    if os.name == "nt" and hasattr(asyncio, "WindowsProactorEventLoopPolicy"):
        try:
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        except Exception:
            pass

    return asyncio.run(
        _scrape_tiktok_keywords_async(
            keywords=keywords,
            max_videos_per_keyword=max_videos_per_keyword,
            scroll_rounds=scroll_rounds,
            scroll_wait_ms=scroll_wait_ms,
            headless=headless,
            nav_timeout_ms=nav_timeout_ms,
            start_date=start_date,
            end_date=end_date,
        )
    )


async def scrape_tiktok_keywords(
    keywords: list[str],
    max_videos_per_keyword: int = 30,
    scroll_rounds: int = 20,
    scroll_wait_ms: int = 1500,
    headless: bool = True,
    nav_timeout_ms: int = 60000,
    start_date: pd.Timestamp | None = None,
    end_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    return await asyncio.to_thread(
        _scrape_tiktok_keywords_sync_runner,
        keywords,
        max_videos_per_keyword,
        scroll_rounds,
        scroll_wait_ms,
        headless,
        nav_timeout_ms,
        start_date,
        end_date,
    )


In [6]:
import sys, asyncio

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [7]:
tt_keywords = _load_keywords_from_csv(TT_KEYWORDS_FILE, column=TT_KEYWORD_COLUMN)
print("Loaded keywords:", len(tt_keywords))
print("Keyword sample:", tt_keywords[:10])

tiktok_df = await scrape_tiktok_keywords(
    keywords=tt_keywords,
    max_videos_per_keyword=TT_MAX_VIDEOS_PER_KEYWORD,
    scroll_rounds=TT_SCROLL_ROUNDS,
    scroll_wait_ms=TT_SCROLL_WAIT_MS,
    headless=TT_HEADLESS,
    nav_timeout_ms=TT_NAV_TIMEOUT_MS,
    start_date=TT_START_DATE,
    end_date=TT_END_DATE,
)

print("Rows collected:", len(tiktok_df))
tiktok_df.head()


Loaded keywords: 70
Keyword sample: ['productive cough', 'cough with phlegm', 'fever', 'high temperature', 'low grade fever', 'sore throat', 'dyspnea', 'difficulty of breathing', 'shortness of breath', 'myalgia']

Scraping keyword: productive cough
  + Kept 0 video(s)

Scraping keyword: cough with phlegm
  + Kept 0 video(s)

Scraping keyword: fever
  + Kept 0 video(s)

Scraping keyword: high temperature
  + Kept 0 video(s)

Scraping keyword: low grade fever
  + Kept 0 video(s)

Scraping keyword: sore throat
  + Kept 0 video(s)

Scraping keyword: dyspnea
  + Kept 0 video(s)

Scraping keyword: difficulty of breathing
  + Kept 0 video(s)

Scraping keyword: shortness of breath
  + Kept 0 video(s)

Scraping keyword: myalgia
  + Kept 0 video(s)

Scraping keyword: body pain
  + Kept 0 video(s)

Scraping keyword: muscle pain
  + Kept 0 video(s)

Scraping keyword: tiredness
  + Kept 0 video(s)

Scraping keyword: fatigue
  + Kept 0 video(s)

Scraping keyword: weakness
  + Kept 0 video(s)

Scrapi

,created_at,id,text,source,url,caption,transcript,like_count,share_count,matched_keyword,author_username


In [ ]:
TT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if tiktok_df.empty:
    tiktok_to_save = tiktok_df.copy()
    skipped_duplicates = 0
else:
    tiktok_to_save = tiktok_df.drop_duplicates(subset=["id"]).copy()
    skipped_duplicates = int(len(tiktok_df) - len(tiktok_to_save))

if TT_OUTPUT_FILE.exists() and not tiktok_to_save.empty:
    try:
        existing_ids = set(
            pd.read_csv(TT_OUTPUT_FILE, usecols=["id"])["id"]
            .dropna()
            .astype(str)
            .tolist()
        )
    except Exception:
        existing_ids = set()

    before = len(tiktok_to_save)
    tiktok_to_save = tiktok_to_save[~tiktok_to_save["id"].astype(str).isin(existing_ids)]
    skipped_duplicates += int(before - len(tiktok_to_save))

if tiktok_to_save.empty:
    print("No new TikTok rows to save.")
else:
    file_exists = TT_OUTPUT_FILE.exists()
    tiktok_to_save.to_csv(
        TT_OUTPUT_FILE,
        mode="a" if file_exists else "w",
        index=False,
        header=not file_exists,
        encoding="utf-8-sig",
    )
    print(f"Saved {len(tiktok_to_save)} row(s) -> {TT_OUTPUT_FILE}")

transcript_present = 0 if tiktok_to_save.empty else int((tiktok_to_save["transcript"].fillna("").astype(str).str.strip() != "").sum())
print(f"Skipped duplicates: {skipped_duplicates}")
print(f"Rows with transcript: {transcript_present}")


In [ ]:
required_cols = ["created_at", "id", "text", "source", "caption", "transcript", "like_count", "share_count"]
missing_cols = [col for col in required_cols if col not in tiktok_df.columns]
if missing_cols:
    raise ValueError(f"Missing required TikTok columns: {missing_cols}")

validation_summary = pd.DataFrame({
    "metric": [
        "rows",
        "duplicate_id_rows",
        "missing_created_at",
        "missing_text",
        "transcript_present_rows",
    ],
    "value": [
        int(len(tiktok_df)),
        int(tiktok_df.duplicated(subset=["id"]).sum()) if not tiktok_df.empty else 0,
        int(tiktok_df["created_at"].isna().sum() + (tiktok_df["created_at"].astype(str).str.strip() == "").sum()) if not tiktok_df.empty else 0,
        int(tiktok_df["text"].isna().sum() + (tiktok_df["text"].astype(str).str.strip() == "").sum()) if not tiktok_df.empty else 0,
        int((tiktok_df["transcript"].fillna("").astype(str).str.strip() != "").sum()) if not tiktok_df.empty else 0,
    ],
})

validation_summary
